In [1]:
#Cell 1 — Title & Imports
# =============================================================
# SALES DEMAND PREDICTION SYSTEM
# Full Pipeline: Data → Features → Train → Predict → Modules
# =============================================================

import sys
import os

# Add project root to path
sys.path.append(os.path.dirname(os.getcwd()))

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

print("✅ Imports successful")
print(f"📁 Working directory: {os.getcwd()}")

✅ Imports successful
📁 Working directory: /home/karthiga/Documents/pythonbykarthiga/sales-demand-system/notebook


In [2]:
#Cell 2 — Generate Dataset
# =============================================================
# STEP 1: GENERATE DATASET
# =============================================================

from src.data.data_generator import generate_dataset

df = generate_dataset()

print(f"\n📊 Dataset Shape : {df.shape}")
print(f"\n📋 Columns       : {list(df.columns)}")
print(f"\n🔍 Sample Rows:")
df.head()

🔄 Generating dataset...
✅ Dataset generated: 2000 rows
📁 Saved to: /home/karthiga/Documents/pythonbykarthiga/sales-demand-system/data/dataset.csv

📊 Sample Data:
         date  day_of_week     item  is_weekend festival  is_festival weather_type location_type time_slot   special_event  is_special_event  is_offer  is_nonveg_day  actual_sales  previous_day_sales
0  2023-01-01            6  Biryani           1     None            0        Sunny   Office_Area    Dinner  Local Festival                 1         0              1            88                   0
1  2023-01-02            0  Biryani           0      Eid            1       Cloudy   Office_Area    Dinner   Cricket Match                 1         1              0            63                  88
2  2023-01-03            1  Biryani           0     None            0        Rainy        Market     Lunch  Local Festival                 1         0              1            93                  63

📊 Dataset Shape : (2000, 15)

📋 Colum

,date,day_of_week,item,is_weekend,festival,is_festival,weather_type,location_type,time_slot,special_event,is_special_event,is_offer,is_nonveg_day,actual_sales,previous_day_sales
0,2023-01-01,6,Biryani,1,None,0,Sunny,Office_Area,Dinner,Local Festival,1,0,1,88,0
1,2023-01-02,0,Biryani,0,Eid,1,Cloudy,Office_Area,Dinner,Cricket Match,1,1,0,63,88
2,2023-01-03,1,Biryani,0,None,0,Rainy,Market,Lunch,Local Festival,1,0,1,93,63
3,2023-01-04,2,Biryani,0,None,0,Windy,Office_Area,Dinner,None,0,1,1,59,93
4,2023-01-05,3,Biryani,0,Eid,1,Sunny,Office_Area,Dinner,Cricket Match,1,0,0,65,59


In [3]:
#Cell 3 — Explore Dataset
# =============================================================
# STEP 2: EXPLORE DATASET
# =============================================================

print("📊 DATASET STATISTICS")
print("="*50)
print(f"  Total Rows     : {len(df)}")
print(f"  Total Columns  : {len(df.columns)}")
print(f"  Date Range     : {df['date'].min()} → {df['date'].max()}")
print(f"  Items          : {df['item'].unique()}")
print(f"  Weather Types  : {df['weather_type'].unique()}")
print(f"  Location Types : {df['location_type'].unique()}")
print(f"  Time Slots     : {df['time_slot'].unique()}")

print("\n📈 SALES STATISTICS BY ITEM:")
print(df.groupby("item")["actual_sales"].describe().round(2))

print("\n📈 SALES BY WEATHER:")
print(df.groupby("weather_type")["actual_sales"].mean().round(2))

📊 DATASET STATISTICS
  Total Rows     : 2000
  Total Columns  : 15
  Date Range     : 2023-01-01 → 2024-02-04
  Items          : ['Biryani' 'Dosa' 'Meals' 'Noodles' 'Parotta']
  Weather Types  : ['Sunny' 'Cloudy' 'Rainy' 'Windy']
  Location Types : ['Office_Area' 'Market' 'Residential' 'College']
  Time Slots     : ['Dinner' 'Lunch']

📈 SALES STATISTICS BY ITEM:
         count    mean    std   min     25%    50%     75%    max
item                                                             
Biryani  400.0  125.02  45.00  37.0   91.00  122.0  153.00  277.0
Dosa     400.0  144.15  40.62  51.0  115.00  139.0  169.00  282.0
Meals    400.0  171.58  46.30  70.0  138.00  167.5  199.00  343.0
Noodles  400.0  113.57  32.14  38.0   90.75  112.5  136.00  237.0
Parotta  400.0  140.90  45.97  48.0  108.00  138.5  169.25  284.0

📈 SALES BY WEATHER:
weather_type
Cloudy    143.02
Rainy     119.26
Sunny     161.92
Windy     133.81
Name: actual_sales, dtype: float64


In [4]:
#Cell 4 — Feature Engineering
# =============================================================
# STEP 3: FEATURE ENGINEERING
# =============================================================

from src.features.feature_engineering import run_feature_pipeline

X, y, encoders = run_feature_pipeline()

print(f"\n✅ Feature Matrix Shape : {X.shape}")
print(f"✅ Target Vector Shape  : {y.shape}")
print(f"\n📋 Features Used:")
for i, col in enumerate(X.columns, 1):
    print(f"   {i:2}. {col}")


  FEATURE ENGINEERING PIPELINE
📂 Loading dataset...
✅ Loaded 2000 rows, 15 columns
✅ Encoders saved to: /home/karthiga/Documents/pythonbykarthiga/sales-demand-system/artifacts/encoder.pkl
✅ Features prepared → X: (2000, 12), y: (2000,)

✅ Feature pipeline complete!
   X shape : (2000, 12)
   y shape : (2000,)
   Features : ['is_weekend', 'is_festival', 'is_nonveg_day', 'is_special_event', 'is_offer', 'previous_day_sales', 'weather_type', 'location_type', 'time_slot', 'festival', 'special_event', 'item']

✅ Feature Matrix Shape : (2000, 12)
✅ Target Vector Shape  : (2000,)

📋 Features Used:
    1. is_weekend
    2. is_festival
    3. is_nonveg_day
    4. is_special_event
    5. is_offer
    6. previous_day_sales
    7. weather_type
    8. location_type
    9. time_slot
   10. festival
   11. special_event
   12. item


In [5]:
#Cell 5 — Train Model
# =============================================================
# STEP 4: TRAIN MODEL
# =============================================================

from src.models.train_model import run_training_pipeline

model = run_training_pipeline()

print("\n🎉 Model training complete!")


  FULL TRAINING PIPELINE

  FEATURE ENGINEERING PIPELINE
📂 Loading dataset...
✅ Loaded 2000 rows, 15 columns
✅ Encoders saved to: /home/karthiga/Documents/pythonbykarthiga/sales-demand-system/artifacts/encoder.pkl
✅ Features prepared → X: (2000, 12), y: (2000,)

✅ Feature pipeline complete!
   X shape : (2000, 12)
   y shape : (2000,)
   Features : ['is_weekend', 'is_festival', 'is_nonveg_day', 'is_special_event', 'is_offer', 'previous_day_sales', 'weather_type', 'location_type', 'time_slot', 'festival', 'special_event', 'item']

  MODEL TRAINING
📊 Train size : 1600 rows
📊 Test size  : 400 rows

🔄 Training model...
✅ Training complete!

  MODEL EVALUATION
  MAE  (Mean Abs Error)  : 14.31
  RMSE (Root Mean Sq Er) : 18.47
  R²   (Accuracy Score)  : 0.8380

📊 Feature Importance:
  is_weekend             ████████ 0.1638
  location_type          ███████ 0.1465
  item                   ██████ 0.1250
  previous_day_sales     ████ 0.0991
  weather_type           ████ 0.0839
  time_slot       

In [6]:
#Cell 6 — Verify Artifacts
# =============================================================
# STEP 5: VERIFY SAVED ARTIFACTS
# =============================================================

from src.models.model_utils import verify_artifacts, get_model_info

# Check files exist
all_good = verify_artifacts()

if all_good:
    get_model_info(model)
else:
    print("❌ Artifacts missing — re-run training!")


  ARTIFACT VERIFICATION
  model.pkl   : ✅ Found
  encoder.pkl : ✅ Found

  MODEL INFO
  Type         : RandomForestRegressor
  N Estimators : 500
  Max Depth    : 15
  N Features   : 12


In [7]:
#Cell 7 — Test Predictions
# =============================================================
# STEP 6: TEST PREDICTIONS
# =============================================================

from src.models.predict import predict_all_items

print("🧪 TEST CASE 1: Sunny Weekday — Office Lunch with Offer")
print("-"*50)
predictions_1 = predict_all_items(
    weather_type       = "Sunny",
    location_type      = "Office_Area",
    time_slot          = "Lunch",
    festival           = "None",
    special_event      = "None",
    is_weekend         = 0,
    is_festival        = 0,
    is_nonveg_day      = 1,
    is_special_event   = 0,
    is_offer           = 1,
    previous_day_sales = 90
)

print("\n🧪 TEST CASE 2: Rainy Weekend — Residential Dinner + Diwali")
print("-"*50)
predictions_2 = predict_all_items(
    weather_type       = "Rainy",
    location_type      = "Residential",
    time_slot          = "Dinner",
    festival           = "Diwali",
    special_event      = "None",
    is_weekend         = 1,
    is_festival        = 1,
    is_nonveg_day      = 1,
    is_special_event   = 0,
    is_offer           = 0,
    previous_day_sales = 110
)

print("\n🧪 TEST CASE 3: Cricket Match — College Area Dinner")
print("-"*50)
predictions_3 = predict_all_items(
    weather_type       = "Cloudy",
    location_type      = "College",
    time_slot          = "Dinner",
    festival           = "None",
    special_event      = "Cricket Match",
    is_weekend         = 0,
    is_festival        = 0,
    is_nonveg_day      = 1,
    is_special_event   = 1,
    is_offer           = 1,
    previous_day_sales = 75
)

🧪 TEST CASE 1: Sunny Weekday — Office Lunch with Offer
--------------------------------------------------

  SALES PREDICTION — ALL ITEMS
✅ Model loaded from: /home/karthiga/Documents/pythonbykarthiga/sales-demand-system/artifacts/model.pkl
✅ Encoders loaded from: /home/karthiga/Documents/pythonbykarthiga/sales-demand-system/artifacts/encoder.pkl
  Biryani      → 133 units
  Meals        → 164 units
  Parotta      → 153 units
  Noodles      → 141 units
  Dosa         → 149 units

🧪 TEST CASE 2: Rainy Weekend — Residential Dinner + Diwali
--------------------------------------------------

  SALES PREDICTION — ALL ITEMS
✅ Model loaded from: /home/karthiga/Documents/pythonbykarthiga/sales-demand-system/artifacts/model.pkl
✅ Encoders loaded from: /home/karthiga/Documents/pythonbykarthiga/sales-demand-system/artifacts/encoder.pkl
  Biryani      → 150 units
  Meals        → 164 units
  Parotta      → 154 units
  Noodles      → 147 units
  Dosa         → 160 units

🧪 TEST CASE 3: Cricket Mat

In [8]:
#Cell 8 — External Modules
# =============================================================
# STEP 7: EXTERNAL MODULES DEMO
# =============================================================

from src.external.weather  import get_weather_summary
from src.external.location import get_location_summary
from src.external.events   import get_events_summary
from src.external.offers   import get_offers_summary

get_weather_summary()
get_location_summary()
get_events_summary()
get_offers_summary()


  WEATHER IMPACT SUMMARY
  Sunny    ██████████████████████    1.10  🔺 Boost
  Rainy    ████████████████          0.80  🔻 Drop
  Cloudy   ███████████████████       0.95  🔻 Drop
  Windy    ██████████████████        0.90  🔻 Drop

  LOCATION DEMAND SUMMARY

  📍 Office_Area
     Lunch    ███████████████████       1.30  🔺 Boost
     Dinner   ██████████                0.70  🔻 Drop

  📍 Residential
     Lunch    █████████████             0.90  🔻 Drop
     Dinner   ██████████████████        1.25  🔺 Boost

  📍 College
     Lunch    █████████████████         1.15  🔺 Boost
     Dinner   ███████████████           1.05  🔺 Boost

  📍 Market
     Lunch    ████████████████          1.10  🔺 Boost
     Dinner   ████████████████          1.10  🔺 Boost

  EVENTS IMPACT SUMMARY

  🎉 Festivals:
    Pongal          █████████████████████     1.40
    Diwali          █████████████████████     1.45
    Christmas       ███████████████████       1.30
    New Year        ██████████████████████    1.50
    Eid     

In [9]:
#Cell 9 — Full Pipeline
# =============================================================
# STEP 8: FULL PIPELINE — Production → Loss
# =============================================================

from src.modules.production  import generate_production_plan
from src.modules.inventory   import (
    initialize_inventory,
    update_inventory,
    get_inventory_status,
    check_low_stock
)
from src.modules.pos         import simulate_sales, get_sales_dict
from src.modules.consumption import (
    convert_pos_to_consumption,
    get_consumption_summary,
    sync_with_inventory
)
from src.modules.waste       import calculate_waste, get_waste_summary
from src.modules.loss        import calculate_loss, generate_loss_report

# Context for today
context = {
    "weather_type"  : "Sunny",
    "location_type" : "Office_Area",
    "time_slot"     : "Lunch",
    "festival"      : "None",
    "special_event" : "None",
    "is_offer"      : 1
}

print("🔄 RUNNING FULL PIPELINE...")
print("="*50)

# Step 1: Predictions (reuse predictions_1)
print("\n📌 Step 1: Predictions ✅")

# Step 2: Production
print("\n📌 Step 2: Production Plan")
production_plan = generate_production_plan(predictions_1)

# Step 3: Inventory Init
print("\n📌 Step 3: Initialize Inventory")
inventory = initialize_inventory(production_plan)

# Step 4: POS Sales
print("\n📌 Step 4: POS — Simulate Sales")
pos_records = simulate_sales(inventory, context)
sales_dict  = get_sales_dict(pos_records)

# Step 5: Update Inventory
print("\n📌 Step 5: Update Inventory")
updated_inventory = update_inventory(inventory, sales_dict)

# Step 6: Consumption
print("\n📌 Step 6: Consumption Tracking")
consumption         = convert_pos_to_consumption(pos_records)
consumption_summary = get_consumption_summary(consumption)

# Step 7: Sync
print("\n📌 Step 7: Sync Consumption ↔ Inventory")
sync_status = sync_with_inventory(consumption, updated_inventory)

# Step 8: Waste
print("\n📌 Step 8: Waste Calculation")
waste_records = calculate_waste(production_plan, pos_records)
waste_summary = get_waste_summary(waste_records)

# Step 9: Loss
print("\n📌 Step 9: Loss Calculation")
loss_records = calculate_loss(waste_records)
loss_report  = generate_loss_report(loss_records)

print("\n🎉 FULL PIPELINE COMPLETE!")

🔄 RUNNING FULL PIPELINE...

📌 Step 1: Predictions ✅

📌 Step 2: Production Plan

  PRODUCTION PLAN
  Biryani      Predicted: 133    Buffer: 13.3%  Produce: 151
  Meals        Predicted: 164    Buffer: 11.1%  Produce: 182
  Parotta      Predicted: 153    Buffer: 11.5%  Produce: 171
  Noodles      Predicted: 141    Buffer: 18.4%  Produce: 167
  Dosa         Predicted: 149    Buffer: 12.0%  Produce: 167

📌 Step 3: Initialize Inventory

  INVENTORY INITIALIZED
  Biryani      → 151 units
  Meals        → 182 units
  Parotta      → 171 units
  Noodles      → 167 units
  Dosa         → 167 units

📌 Step 4: POS — Simulate Sales

  POS — SALES SIMULATION
  Biryani      Stock: 151    Sold: 132
  Meals        Stock: 182    Sold: 146
  Parotta      Stock: 171    Sold: 157
  Noodles      Stock: 167    Sold: 154
  Dosa         Stock: 167    Sold: 156

📌 Step 5: Update Inventory

  INVENTORY UPDATE
  Biryani      Before: 151    Sold: 132    Remaining: 19  ✅
  Meals        Before: 182    Sold: 146    R

In [10]:
#Cell 10 — Final Summary
# =============================================================
# STEP 9: FINAL SUMMARY REPORT
# =============================================================

import json

print("\n" + "="*60)
print("        SALES DEMAND SYSTEM — FINAL REPORT")
print("="*60)

print("\n📦 PRODUCTION PLAN:")
for p in production_plan:
    print(f"  {p['item']:<12} → Produce: {p['production_qty']} units")

print("\n🏪 INVENTORY (After Sales):")
for item, qty in updated_inventory.items():
    print(f"  {item:<12} → Remaining: {qty} units")

print("\n🗑️  WASTE SUMMARY:")
waste_data = json.loads(waste_summary)
print(f"  Total Produced : {waste_data['total_produced']}")
print(f"  Total Sold     : {waste_data['total_sold']}")
print(f"  Total Waste    : {waste_data['total_waste']}")
print(f"  Waste %        : {waste_data['overall_waste_pct']}%")

print("\n💸 LOSS REPORT:")
loss_data = json.loads(loss_report)
print(f"  Total Loss     : ₹{loss_data['total_loss']}")
print(f"  Severity       : {loss_data['severity']}")

print("\n" + "="*60)
print("  ✅ System working end-to-end successfully!")
print("="*60)


        SALES DEMAND SYSTEM — FINAL REPORT

📦 PRODUCTION PLAN:
  Biryani      → Produce: 151 units
  Meals        → Produce: 182 units
  Parotta      → Produce: 171 units
  Noodles      → Produce: 167 units
  Dosa         → Produce: 167 units

🏪 INVENTORY (After Sales):
  Biryani      → Remaining: 19 units
  Meals        → Remaining: 36 units
  Parotta      → Remaining: 14 units
  Noodles      → Remaining: 13 units
  Dosa         → Remaining: 11 units

🗑️  WASTE SUMMARY:
  Total Produced : 838
  Total Sold     : 745
  Total Waste    : 93
  Waste %        : 11.1%

💸 LOSS REPORT:
  Total Loss     : ₹7210
  Severity       : 🔴 High Loss

  ✅ System working end-to-end successfully!


Cell 1  → Imports
Cell 2  → Generate Dataset
Cell 3  → Explore Dataset
Cell 4  → Feature Engineering
Cell 5  → Train Model
Cell 6  → Verify Artifacts
Cell 7  → Test Predictions (3 scenarios)
Cell 8  → External Modules Demo
Cell 9  → Full Pipeline
Cell 10 → Final Summary Report

In [11]:
# =============================================================
# STEP 10: LOSS IMPROVEMENT ANALYSIS
# =============================================================

print("="*60)
print("   LOSS IMPROVEMENT ANALYSIS")
print("="*60)

print("""
❌ PROBLEM IDENTIFIED:
   Total Loss    : ₹7210
   Severity      : 🔴 High Loss
   Root Cause    : Buffer too high (10-20%) causing overproduction
   Worst Items   : Meals (₹2880), Biryani (₹2280)
""")

# -------------------------------------------------------
# IMPROVEMENT 1: Reduce Buffer Based on Waste %
# -------------------------------------------------------
print("✅ IMPROVEMENT 1: Smart Buffer (based on waste %)")
print("-"*50)

improved_plan = []
for record in waste_records:
    item       = record["item"]
    waste_pct  = record["waste_pct"]
    predicted  = next(
        p["predicted_sales"] for p in production_plan
        if p["item"] == item
    )

    # Smart buffer: if waste is high → reduce buffer
    if waste_pct > 15:
        smart_buffer = 0.05   # only 5% buffer
    elif waste_pct > 10:
        smart_buffer = 0.08   # 8% buffer
    else:
        smart_buffer = 0.10   # keep 10%

    new_qty = round(predicted * (1 + smart_buffer))
    improved_plan.append({
        "item"         : item,
        "old_produce"  : record["produced"],
        "new_produce"  : new_qty,
        "old_waste"    : record["waste"],
        "new_waste"    : max(0, new_qty - record["sold"]),
    })

    print(f"  {item:<12} "
          f"Buffer: {int(smart_buffer*100)}%  "
          f"Old Qty: {record['produced']}  "
          f"New Qty: {new_qty}")

# -------------------------------------------------------
# IMPROVEMENT 2: Calculate New Loss
# -------------------------------------------------------
print("\n✅ IMPROVEMENT 2: Recalculated Loss After Smart Buffer")
print("-"*50)

from src.config.settings import ITEM_COST

old_total_loss = 7210
new_total_loss = 0

for p in improved_plan:
    item      = p["item"]
    new_waste = p["new_waste"]
    cost      = ITEM_COST.get(item, 0)
    new_loss  = new_waste * cost
    new_total_loss += new_loss
    print(f"  {item:<12} "
          f"New Waste: {new_waste:<6} × "
          f"₹{cost:<6} = ₹{new_loss}")

# -------------------------------------------------------
# IMPROVEMENT 3: Comparison Report
# -------------------------------------------------------
saved      = old_total_loss - new_total_loss
saved_pct  = round((saved / old_total_loss) * 100, 1)

print("\n" + "="*60)
print("   BEFORE vs AFTER COMPARISON")
print("="*60)
print(f"  ❌ Old Loss  : ₹{old_total_loss}  🔴 High Loss")
print(f"  ✅ New Loss  : ₹{new_total_loss}  🟢 Reduced")
print(f"  💰 Saved     : ₹{saved} ({saved_pct}% reduction)")
print("="*60)

severity = (
    "🟢 Acceptable" if new_total_loss < 500  else
    "🟡 Moderate"   if new_total_loss < 1500 else
    "🔴 High Loss"
)
print(f"\n  New Severity : {severity}")
print("\n  📌 Recommendation:")
print("     → Use smart buffer (5-8%) for high-waste items")
print("     → Use 10% buffer only for low-waste items")
print("     → Re-train model weekly with fresh POS data")
print("="*60)

   LOSS IMPROVEMENT ANALYSIS

❌ PROBLEM IDENTIFIED:
   Total Loss    : ₹7210
   Severity      : 🔴 High Loss
   Root Cause    : Buffer too high (10-20%) causing overproduction
   Worst Items   : Meals (₹2880), Biryani (₹2280)

✅ IMPROVEMENT 1: Smart Buffer (based on waste %)
--------------------------------------------------
  Biryani      Buffer: 8%  Old Qty: 151  New Qty: 144
  Meals        Buffer: 5%  Old Qty: 182  New Qty: 172
  Parotta      Buffer: 10%  Old Qty: 171  New Qty: 168
  Noodles      Buffer: 10%  Old Qty: 167  New Qty: 155
  Dosa         Buffer: 10%  Old Qty: 167  New Qty: 164

✅ IMPROVEMENT 2: Recalculated Loss After Smart Buffer
--------------------------------------------------
  Biryani      New Waste: 12     × ₹120    = ₹1440
  Meals        New Waste: 26     × ₹80     = ₹2080
  Parotta      New Waste: 11     × ₹50     = ₹550
  Noodles      New Waste: 1      × ₹70     = ₹70
  Dosa         New Waste: 8      × ₹40     = ₹320

   BEFORE vs AFTER COMPARISON
  ❌ Old Loss 